# RK1828 FastSAM-s 实例分割 Demo

[FastSAM](https://github.com/CASIA-IVA-Lab/FastSAM)（中科院自动化所）是基于 YOLOv8-seg 的
**可提示实例分割**模型。本 demo 用 "segment everything" 模式：一次前向找出图中所有物体并给出掩码。

模型两个输出：

| 输出 | 形状 | 含义 |
|---|---|---|
| `output0` | `[1,37,8400]` | cx, cy, w, h, 置信度 + 32 个 mask 系数（已解码） |
| `output1` | `[1,32,160,160]` | mask prototypes（掩码基） |

后处理：置信度过滤 → NMS → `sigmoid(系数 @ prototypes)` 得到每个物体的掩码。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 选择图片 |
| 3 | 前处理（letterbox） |
| 4 | RKNN 推理 |
| 5 | 后处理（NMS + 掩码合成）与可视化 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


本 demo 下载 FastSAM-s 模型和示例图片。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("fastsam-s-rknn3-rk1828", "fastsam_s.rknn", "models/fastsam/fastsam_s.rknn"),
    ("fastsam-s-rknn3-rk1828", "fastsam_s.weight", "models/fastsam/fastsam_s.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "datasets/bus.jpg"),
    ("mobilenet_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 选择图片

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="图片:", layout=widgets.Layout(width="60%"))
w_conf = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05,
                             description="置信度:", style={"description_width": "80px"})
display(w_image, w_conf)
print("选好后继续往下运行 ▼")

## 3. 前处理：letterbox

In [ ]:
from PIL import Image
from vision_post import letterbox, to_input_array

img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"原图 {img.size} -> 输入 {x.shape}")
img_lb

## 4. RKNN 推理

In [ ]:
import time
from rknn3 import Rknn3Model

model = Rknn3Model("models/fastsam/fastsam_s.rknn")
print("输出:", model.output_attrs)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"推理耗时: {(time.time()-t0)*1000:.1f} ms")

## 5. 后处理：NMS + 掩码合成

`sigmoid(mask系数 @ prototypes)` 得到 160×160 掩码，放大到输入尺寸、按框裁剪，
再映射回原图；过滤掉过小（面积占比 < 0.2%）的碎片掩码。

In [ ]:
import numpy as np
from vision_post import decode_fastsam, scale_masks, unletterbox_boxes

boxes, scores, masks = decode_fastsam(outputs, conf_thresh=w_conf.value)
masks_full = scale_masks(masks, boxes, (img.height, img.width), SCALE, PAD_X, PAD_Y)
boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)

ratio = (masks_full > 0.5).mean(axis=(1, 2))
keep = ratio > 0.002
boxes, scores, masks_full = boxes[keep], scores[keep], masks_full[keep]
print(f"共 {len(boxes)} 个物体（过滤碎片后）")

## 6. 可视化

In [ ]:
from vision_post import draw_boxes, draw_masks

overlay = draw_masks(img, masks_full)
result = draw_boxes(overlay, boxes, np.zeros(len(boxes)), scores)
model.close()
result

## 小结

- FastSAM 是**类别无关**的：只输出"是不是物体"和掩码，不分类别。
- 掩码 = `sigmoid(32 个系数 × 32 通道 prototypes)`，这是 YOLOv8-seg 系列的通用做法。